# Task 6 — Causal Inference: Does Resolution Time Affect Severity or Defect Category?

This notebook investigates whether **Resolution Time** has a meaningful relationship
with **Severity Category** and **Defect Category**, and simulates how a reduction in
resolution time might affect these outcomes.

**Important framing note:** this dataset is observational (not from a designed
experiment), so we cannot definitively prove *causation* — only measure *association*
and reason carefully about what it might imply. We are explicit about this distinction
throughout, rather than overclaiming a causal effect the data can't actually support.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Load the same source data used by the rest of the application, so this
# notebook's findings stay consistent with the dashboard.
df = pd.read_excel("../backend/data/quality_data.xlsx")

# Same cleanup step as backend/app/data.py: ensure resolution time is
# numeric (it may load as 'object' dtype if the column has mixed types).
df["Time to Resolution (days)"] = pd.to_numeric(
    df["Time to Resolution (days)"], errors="coerce"
)

df.rename(columns={
    "Time to Resolution (days)": "resolution_time_days",
    "Severity Category": "severity_category",
    "Defect Category": "defect_category",
}, inplace=True)

print(f"Total records: {len(df)}")
print(f"Records with a recorded resolution time: {df['resolution_time_days'].notna().sum()}")

Total records: 9000
Records with a recorded resolution time: 6790


In [2]:
resolved = df[df["resolution_time_days"].notna()].copy()

print("=== Resolution time (days) by Severity Category ===")
display(resolved.groupby("severity_category")["resolution_time_days"].agg(["count", "mean", "std"]).round(2))

print("\n=== Resolution time (days) by Defect Category ===")
display(resolved.groupby("defect_category")["resolution_time_days"].agg(["count", "mean", "std"]).round(2))

=== Resolution time (days) by Severity Category ===


,count,mean,std
severity_category,,,
Extreme - major functional risk,1549,1.41,4.71
Extreme - safety/legal/homologation,473,1.41,5.11
Extreme - safety/legal/inoperable,109,1.03,2.25
Medium - ASAP dealership,1760,1.47,5.17
Medium - immediate dealership,1989,1.28,4.52
Medium - next service visit,910,1.20,4.17



=== Resolution time (days) by Defect Category ===


,count,mean,std
defect_category,,,
Cosmetic,2603,1.33,4.70
Critical,9,1.53,3.61
Electrical,2669,1.34,4.70
Functional,1450,1.42,4.85
Safety,57,1.07,2.54
Structural,2,0.30,0.02


In [3]:
# We use the Kruskal-Wallis test — a non-parametric alternative to
# one-way ANOVA — because resolution_time_days is heavily right-skewed
# (many quick resolutions, a long tail of slow ones), which violates the
# "normally distributed" assumption ANOVA relies on. Kruskal-Wallis
# instead compares the RANKS of values across groups, making it robust
# to this kind of skew.

groups_severity = [g["resolution_time_days"].values for _, g in resolved.groupby("severity_category")]
h_stat_sev, p_value_sev = stats.kruskal(*groups_severity)

print("Kruskal-Wallis test: resolution_time_days across severity_category")
print(f"H-statistic: {h_stat_sev:.3f}, p-value: {p_value_sev:.4f}")

# For defect_category, we exclude groups with fewer than 5 records
# (Structural, n=2) since a group that small can't meaningfully
# participate in a rank-based test — its result would be noise.
groups_defect = [
    g["resolution_time_days"].values
    for name, g in resolved.groupby("defect_category")
    if len(g) >= 5
]
h_stat_def, p_value_def = stats.kruskal(*groups_defect)

print("\nKruskal-Wallis test: resolution_time_days across defect_category (groups with n>=5)")
print(f"H-statistic: {h_stat_def:.3f}, p-value: {p_value_def:.4f}")

print("\nInterpretation: a p-value below 0.05 would suggest a statistically")
print("significant difference between groups. Values above 0.05 mean we")
print("cannot conclude the groups genuinely differ — the small differences")
print("observed above are consistent with random variation.")

Kruskal-Wallis test: resolution_time_days across severity_category
H-statistic: 3.949, p-value: 0.5568

Kruskal-Wallis test: resolution_time_days across defect_category (groups with n>=5)
H-statistic: 4.681, p-value: 0.3217

Interpretation: a p-value below 0.05 would suggest a statistically
significant difference between groups. Values above 0.05 mean we
cannot conclude the groups genuinely differ — the small differences
observed above are consistent with random variation.


In [4]:
# We now test the relationship a second way: instead of comparing group
# averages, we fit a model that tries to PREDICT severity from resolution
# time directly, and check how well it actually performs.
#
# We collapse severity_category into a binary outcome (Extreme vs Medium)
# rather than treating severity_rating as a numeric scale — recall from
# the dashboard analysis that severity_rating (1-6) is NOT an ordered
# scale, so using it as a continuous numeric predictor/outcome would be
# statistically meaningless. severity_category, on the other hand, is the
# dataset's real, meaningful classification.

resolved["is_extreme"] = resolved["severity_category"].str.startswith("Extreme").astype(int)

X = resolved[["resolution_time_days"]].values
y = resolved["is_extreme"].values

model = LogisticRegression()
model.fit(X, y)

pred_proba = model.predict_proba(X)[:, 1]
auc = roc_auc_score(y, pred_proba)

print("Logistic regression: is_extreme ~ resolution_time_days")
print(f"Coefficient: {model.coef_[0][0]:.5f}")
print(f"AUC (model quality): {auc:.4f}")
print("(AUC of 0.50 = no predictive power at all, i.e. equivalent to a coin flip;")
print(" AUC of 1.00 = perfect prediction)")
print(f"\nBase rate of 'Extreme' classification: {y.mean():.2%}")

Logistic regression: is_extreme ~ resolution_time_days
Coefficient: 0.00243
AUC (model quality): 0.5078
(AUC of 0.50 = no predictive power at all, i.e. equivalent to a coin flip;
 AUC of 1.00 = perfect prediction)

Base rate of 'Extreme' classification: 31.38%


In [5]:
# Now we simulate the case study's specific ask: "how would an X%
# reduction in Resolution Time affect Severity Rating and Defect
# Category?" We take the ACTUAL resolution times in the dataset, reduce
# every one of them by a percentage, and see what the fitted model would
# then predict.
#
# IMPORTANT CAVEAT: this simulation shows what our model WOULD predict
# under a hypothetical reduction — it is NOT proof that reducing
# resolution time actually causes any real-world change, since the model
# itself found no meaningful relationship to begin with (AUC ~ 0.51).
# We include this simulation because the case study explicitly asks for
# it, but the honest conclusion (see below) is that this simulation shows
# a hypothetical intervention would have almost no predicted effect,
# which is itself the answer to the question.

print(f"Original mean resolution time: {resolved['resolution_time_days'].mean():.2f} days")
print(f"Original average predicted P(Extreme): {pred_proba.mean():.4f}")
print()

for pct_reduction in [10, 25, 50]:
    reduced_X = X * (1 - pct_reduction / 100)
    reduced_proba = model.predict_proba(reduced_X)[:, 1]
    change = reduced_proba.mean() - pred_proba.mean()
    print(
        f"{pct_reduction}% reduction in resolution time -> "
        f"predicted P(Extreme): {reduced_proba.mean():.4f} "
        f"(change: {change:+.4f}, i.e. {change*100:+.2f} percentage points)"
    )

Original mean resolution time: 1.35 days
Original average predicted P(Extreme): 0.3138

10% reduction in resolution time -> predicted P(Extreme): 0.3138 (change: -0.0001, i.e. -0.01 percentage points)
25% reduction in resolution time -> predicted P(Extreme): 0.3137 (change: -0.0002, i.e. -0.02 percentage points)
50% reduction in resolution time -> predicted P(Extreme): 0.3135 (change: -0.0004, i.e. -0.04 percentage points)


## Interpretation & Decision Relevance

**Finding:** Across three independent checks — group-level averages, a formal
Kruskal-Wallis test, and a logistic regression with simulated interventions — we
found **no meaningful relationship** between Resolution Time and either Severity
Category or Defect Category in this dataset:

- Average resolution times are nearly identical across all severity and defect
  categories (~1.0–1.5 days), and the Kruskal-Wallis test could not rule out that
  these small differences are due to random chance (p = 0.56 and p = 0.32,
  both far above the 0.05 significance threshold).
- A logistic regression predicting "Extreme" severity from resolution time
  performed no better than random guessing (AUC = 0.51).
- Simulating a 10%, 25%, or even 50% reduction in resolution time changed the
  predicted probability of "Extreme" classification by less than 0.05 percentage
  points — an effectively negligible shift.

**What this means for decision-making:** if BMW's quality team were considering
investing in initiatives specifically to *speed up defect resolution* as a way to
*reduce the severity or type of defects reported*, this analysis suggests that
approach would not be effective — resolution speed and defect severity/type
appear to be **independent** in this data. Severity and category are likely
determined by the nature of the defect itself (e.g. what broke, and how) rather
than how quickly it happens to get fixed afterward. Any resources aimed at
improving severity outcomes would likely be better directed elsewhere — for
example, upstream process or design changes, rather than resolution-time
initiatives.

**A note on causation:** as stated at the start of this notebook, this dataset is
observational, not experimental. We can confidently say resolution time and
severity/category are *not associated* in this data — but we cannot use this
analysis to make causal claims about *any* variable, only to responsibly rule out
one specific hypothesized relationship. A true causal study would require a
controlled or quasi-experimental design (e.g. comparing resolution times before
and after a specific process change, holding other factors constant).

**Data limitation worth noting:** the "Structural" (n=2) and "Critical" (n=9)
defect categories have very small sample sizes, limiting how much confidence we
can place in their specific averages — though this does not affect the overall
conclusion, which is driven primarily by the much larger categories.